### Quickstart: Compare runs, choose a model, and deploy it to a REST API

In this quickstart, you will:

- Run a hyperparameter sweep on a training script

- Compare the results of the runs in the MLflow UI

- Choose the best run and register it as a model

- Deploy the model to a REST API

- Build a container image suitable for deployment to a cloud platform


In [1]:
import keras
import numpy as np
import pandas as pd
import trials
from hyperopt import STATUS_OK,Trials,fmin,hp,tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

F:\Codes\Tutorials\MLFlowStarter\.venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
F:\Codes\Tutorials\MLFlowStarter\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the dataset
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [3]:
# Split the data into training, validation and test sets

train,test = train_test_split(data,test_size=0.25,random_state=42)
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
2835,6.3,0.25,0.22,3.30,0.048,41.0,161.0,0.99256,3.16,0.50,10.5,6
1157,7.8,0.30,0.29,16.85,0.054,23.0,135.0,0.99980,3.16,0.38,9.0,6
744,7.4,0.38,0.27,7.50,0.041,24.0,160.0,0.99535,3.17,0.43,10.0,5
1448,7.4,0.16,0.49,1.20,0.055,18.0,150.0,0.99170,3.23,0.47,11.2,6
3338,7.2,0.27,0.28,15.20,0.046,6.0,41.0,0.99665,3.17,0.39,10.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [4]:
train_x=train.drop(['quality'],axis=1).values
train_y=train[['quality']].values.ravel()

# test dataset
test_x=test.drop(['quality'],axis=1).values
test_y=test[['quality']].values.ravel()

# splitting this train data into train and validation

train_x,valid_x,train_y,valid_y=train_test_split(train_x,train_y,test_size=0.20,random_state=42)

signature=infer_signature(train_x,train_y)

In [5]:
# ANN Model

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):

    # Define model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)

    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(1)
        ]
    )

    # compile the model
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params["lr"],
        momentum=params["momentum"]
    ),
    loss="mean_squared_error",
    metrics=[keras.metrics.RootMeanSquaredError()]
    )

    # Train the ANN model with lr and momentum params with MLFLOW tracking
    with mlflow.start_run(nested=True):
        model.fit(train_x, train_y, validation_data=(valid_x, valid_y), epochs=epochs, batch_size=64)

        # Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64)

        eval_rmse = eval_result[1]

        # Log the parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        # Log the model
        mlflow.tensorflow.log_model(model, "model", signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [6]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [7]:
space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0)
}

In [8]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment("/wine-quality")

with mlflow.start_run():
    # conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn = objective,
        space = space,
        algo = tpe.suggest,
        max_evals=4,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

2026/04/20 14:57:48 INFO mlflow.tracking.fluent: Experiment with name '/wine-quality' does not exist. Creating a new experiment.


  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.
Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 17s 380ms/step - loss: 36.6977 - root_mean_squared_error: 6.0579
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 32.3209 - root_mean_squared_error: 5.6851 - val_loss: 28.2687 - val_root_mean_squared_error: 5.3168

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 28.5222 - root_mean_squared_error: 5.3406
42/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 26.7322 - root_mean_squared_error: 5.1697 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 25.3092 - root_mean_squared_error: 5.0308 - val_loss: 22.0802 - val_root_mean_squared_error: 4.6990

Epoch 3/3                                            

 1/46 

2026/04/20 14:57:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run debonair-rat-839 at: http://127.0.0.1:5000/#/experiments/3/runs/9ab26357927e4bd3b62d2f6079a300cd

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 14s 313ms/step - loss: 35.9990 - root_mean_squared_error: 5.9999
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 20.5071 - root_mean_squared_error: 4.5285 - val_loss: 9.3587 - val_root_mean_squared_error: 3.0592

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 9.2238 - root_mean_squared_error: 3.0371
42/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 7.2236 - root_mean_squared_error: 2.6793 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.3814 - root_mean_squared_error: 2.3198 - val_loss: 3.1248 - val_root_mean_squared_error: 1.7677

Epoch 3/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 

2026/04/20 14:58:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



...

🏃 View run skittish-donkey-282 at: http://127.0.0.1:5000/#/experiments/3/runs/9967a3d840de463c85057a197ae6ef49

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 17s 388ms/step - loss: 36.1536 - root_mean_squared_error: 6.0128
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 21.2555 - root_mean_squared_error: 4.6104 - val_loss: 10.4373 - val_root_mean_squared_error: 3.2307

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 10.9769 - root_mean_squared_error: 3.3131
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.3758 - root_mean_squared_error: 2.5250 - val_loss: 3.5631 - val_root_mean_squared_error: 1.8876

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 3.1734 - root_mean_squared_error: 1.7814


2026/04/20 14:58:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run exultant-tern-622 at: http://127.0.0.1:5000/#/experiments/3/runs/ec40cc433889442dbf5e05a55929881e

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3                   

100%|██████████| 4/4 [00:39<00:00,  9.93s/trial, best loss: 1.4998432397842407]

2026/04/20 14:58:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.0012297476850037766), 'momentum': np.float64(0.21615216725449238)}
Best eval rmse: 1.4998432397842407
🏃 View run flawless-finch-660 at: http://127.0.0.1:5000/#/experiments/3/runs/1fdd75bc2dee4533933fb293f2bebde8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [10]:
# Inferencing

from mlflow.models import validate_serving_input

model_uri = 'runs:/c0891c8f07a240f1b8ff9c50f731d68a/model'

# The logged model does not contain an input_example.
# Manually generate a serving payload to verify your model prior to deployment.
from mlflow.models import convert_input_example_to_serving_input

# Define INPUT_EXAMPLE via assignment with your own input example to the model
# A valid input example is a data instance suitable for pyfunc prediction
serving_payload = convert_input_example_to_serving_input(test_x)

# Validate the serving payload works on the model
validate_serving_input(model_uri, serving_payload)

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


array([[4.4089937],
       [7.1923037],
       [5.464072 ],
       ...,
       [6.542717 ],
       [7.146001 ],
       [4.38675  ]], shape=(1225, 1), dtype=float32)

In [11]:
# Load model as a PyFuncModel.
model_uri = 'runs:/c0891c8f07a240f1b8ff9c50f731d68a/model'
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Predict on a Pandas DataFrame.
import pandas as pd
loaded_model.predict(pd.DataFrame(test_x))

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


array([[4.4089937],
       [7.1923037],
       [5.464072 ],
       ...,
       [6.542717 ],
       [7.146001 ],
       [4.38675  ]], shape=(1225, 1), dtype=float32)

In [12]:
# Register in the model registry
mlflow.register_model(model_uri,"wine-quality")

Successfully registered model 'wine-quality'.
2026/04/20 15:24:23 WARNING mlflow.tracking._model_registry.fluent: Run with id c0891c8f07a240f1b8ff9c50f731d68a has no artifacts at artifact path 'model', registering model based on models:/m-ae25ab59fdb749db9b072dabaa2dab48 instead
2026/04/20 15:24:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine-quality, version 1
Created version '1' of model 'wine-quality'.


<ModelVersion: aliases=[], creation_timestamp=1776678863936, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1776678863936, metrics=None, model_id=None, name='wine-quality', params=None, run_id='c0891c8f07a240f1b8ff9c50f731d68a', run_link='', source='models:/m-ae25ab59fdb749db9b072dabaa2dab48', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>